In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kgan31/hindi-poems/kavitas_cleaned_processed.csv
/kaggle/input/datasets/kgan31/hindi-poems/kavitas_remaining_processed.csv
/kaggle/input/datasets/kgan31/complete-kavita-dataset/kavitas_merged.csv


In [2]:
!pip install -q transformers datasets accelerate numpy pandas tqdm huggingface_hub

In [4]:
import pandas as pd
import numpy as np
import random
import torch
import os
from tqdm.auto import tqdm
from datasets import Dataset, load_dataset
from huggingface_hub import login, HfApi
from kaggle_secrets import UserSecretsClient
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)

# -------------------------------------------------------------------
# 0. HUGGING FACE AUTHENTICATION & SETUP
# -------------------------------------------------------------------

try:
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Successfully logged into Hugging Face Hub!")
except Exception as e:
    print(f"Error accessing HF_TOKEN: {e}")
    print("Please ensure you have added 'HF_TOKEN' to your Kaggle Secrets.")
    HF_TOKEN = None

# Set your Hugging Face username and repo names
HF_USERNAME = "KGan31" 
MODEL_REPO_NAME = "Doha-Gen"
HUB_MODEL_ID = f"{HF_USERNAME}/{MODEL_REPO_NAME}"
DATASET_REPO_ID = f"{HF_USERNAME}/{MODEL_REPO_NAME}-Dataset" # Distinct name for the dataset

MODEL_NAME = "google/byt5-small"

# -------------------------------------------------------------------
# 1. CHECK IF TOKENIZED DATASET EXISTS ON HUGGING FACE
# -------------------------------------------------------------------

api = HfApi()
try:
    # Try to fetch dataset info. If it doesn't exist, it throws an error.
    api.dataset_info(DATASET_REPO_ID, token=HF_TOKEN)
    dataset_exists = True
    print(f"\n✅ Found existing dataset at {DATASET_REPO_ID}!")
except Exception:
    dataset_exists = False
    print(f"\n⚠️ Dataset {DATASET_REPO_ID} not found. Will generate it from scratch.")

# We always need the tokenizer initialized
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if dataset_exists:
    # --- LOAD DIRECTLY FROM HUGGING FACE ---
    print("Downloading tokenized dataset from Hugging Face Hub...")
    tokenized_datasets = load_dataset(DATASET_REPO_ID, token=HF_TOKEN)
    print("Dataset loaded successfully! Skipping local tokenization.")

else:
    # --- PROCESS FROM SCRATCH ---
    print("Starting local data processing pipeline...")
    tqdm.pandas(desc="Applying Noise to Poems")

    def apply_noise(text):
        if pd.isna(text) or not text.strip():
            return ""
        
        # A. Sentence (Line) Permutation
        lines = [line.strip() for line in str(text).split('\\') if line.strip()]
        random.shuffle(lines)
        shuffled_text = " ".join(lines)
        
        # B. Span Masking
        words = shuffled_text.split()
        if not words:
            return text
            
        num_words = len(words)
        num_to_mask = int(0.3 * num_words)
        masked_indices = set()
        
        while len(masked_indices) < num_to_mask:
            span_length = np.random.poisson(lam=3)
            span_length = max(1, span_length) 
            start_idx = random.randint(0, num_words - 1)
            
            for i in range(start_idx, min(start_idx + span_length, num_words)):
                if len(masked_indices) < num_to_mask:
                    masked_indices.add(i)
                else:
                    break

        corrupted_words = []
        i = 0
        while i < num_words:
            if i in masked_indices:
                corrupted_words.append("[MASK]")
                while i < num_words and i in masked_indices:
                    i += 1
            else:
                corrupted_words.append(words[i])
                i += 1
                
        return " ".join(corrupted_words)

    # Load Dataset
    csv_path = '/kaggle/input/datasets/kgan31/complete-kavita-dataset/kavitas_merged.csv'
    print(f"Loading data from {csv_path}...")
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=['kavita_text'])
    df = df.head(20000)

    print("Corrupting text to create pretraining inputs...")
    df['corrupted_text'] = df['kavita_text'].progress_apply(apply_noise)

    dataset = Dataset.from_pandas(df[['corrupted_text', 'kavita_text']])
    dataset = dataset.train_test_split(test_size=0.05)

    # Tokenize
    MAX_LENGTH = 1024 
    def preprocess_function(examples):
        model_inputs = tokenizer(
            examples["corrupted_text"], 
            max_length=MAX_LENGTH, 
            truncation=True
        )
        labels = tokenizer(
            text_target=examples["kavita_text"], 
            max_length=MAX_LENGTH, 
            truncation=True
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    print("Tokenizing datasets...")
    NUM_CORES = os.cpu_count() or 2 
    tokenized_datasets = dataset.map(
        preprocess_function, 
        batched=True, 
        num_proc=NUM_CORES, 
        remove_columns=dataset["train"].column_names,
        desc="Tokenizing Data"
    )

    # Push to Hugging Face
    if HF_TOKEN:
        print(f"Pushing tokenized dataset to Hugging Face: {DATASET_REPO_ID}...")
        tokenized_datasets.push_to_hub(DATASET_REPO_ID, token=HF_TOKEN)
        print("Dataset successfully uploaded!")


# -------------------------------------------------------------------
# 2. MODEL INITIALIZATION, TRAINING & UPLOAD
# -------------------------------------------------------------------

print("\nInitializing model for training...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Silence the tied weights warning
model.config.tie_word_embeddings = False 

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/byt5-hindi-poem-denoiser",
    eval_strategy="epoch",
    learning_rate=3e-4,
    
    # --- MEMORY FIXES ---
    per_device_train_batch_size=2, # Reduced from 4 to 2
    per_device_eval_batch_size=2,  # Reduced from 4 to 2
    gradient_accumulation_steps=2, # Simulates a batch size of 4 (2x2) without using extra memory
    fp16=True,                     # Enable Mixed Precision (Uses half the memory!)
    # --------------------
    
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3, 
    predict_with_generate=True,
    logging_steps=50,
    report_to="none",
    push_to_hub=True if HF_TOKEN else False,
    hub_model_id=HUB_MODEL_ID if HF_TOKEN else None,
    hub_strategy="end" 
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Starting training (progress bar will appear below)...")
trainer.train()

trainer.save_model("/kaggle/working/byt5-hindi-poem-adapted")

if HF_TOKEN:
    print(f"Pushing final model to Hugging Face Hub: https://huggingface.co/{HUB_MODEL_ID}")
    trainer.push_to_hub(commit_message="Completed ByT5 Stage 1 Continued Pretraining on Hindi Poems")
    print("Upload complete!")
else:
    print("Training complete! Model saved locally to /kaggle/working/")

Successfully logged into Hugging Face Hub!

✅ Found existing dataset at KGan31/Doha-Gen-Dataset!
Dataset loaded successfully! Skipping local tokenization.

Initializing model for training...


Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Starting training (progress bar will appear below)...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Upload complete!
